# Solving the Neoclassical Growth Model with Neural Networks

This notebook demonstrates a deep learning approach to solving the continuous-time neoclassical growth model. We parameterize the value function as a neural network and train it by minimizing violations of the Hamilton-Jacobi-Bellman (HJB) equation across the state space.

The key advantage over traditional finite difference methods is scalability: computational cost grows polynomially rather than exponentially with the number of state variables.

**Acknowledgment**: This example is based on code provided by Jesús Fernández-Villaverde, from his *Journal of Economic Literature* review of deep learning methods for solving dynamic economic models.

## 1. The Economic Model

### The Social Planner's Problem

A social planner chooses consumption $C_t$ to maximize:
$$\max_{{\{C_t\}_{{t \geq 0}}}} \int_0^\infty e^{-\rho t} U(C_t) \, dt$$

subject to the capital accumulation equation:
$$\dot{K}_t = Y_t - C_t - \delta K_t$$

where:
- Output is $Y_t = A K_t^\alpha$ (Cobb-Douglas production)
- Utility is CRRA: $U(C) = \frac{C^{1-\gamma}}{1-\gamma}$
- $\gamma$ is the coefficient of relative risk aversion
- $\rho$ is the discount rate (rate of time preference)
- $\delta$ is the capital depreciation rate
- $A$ is total factor productivity (TFP)
- $\alpha$ is the capital share of income

### The Hamilton-Jacobi-Bellman Equation

Let $V(k)$ denote the value function, where $k = K/L$ is capital per unit of labor (in this setup, labor is constant). The HJB equation is:
$$\rho V(k) = \max_c \{U(c) + V'(k) [Ak^\alpha - c - \delta k]\}$$

The first-order condition with respect to consumption gives:
$$U'(c) = V'(k)$$

Since $U'(c) = c^{-\gamma}$, the optimal consumption policy is:
$$c^*(k) = [V'(k)]^{-1/\gamma}$$

Substituting back into the HJB equation, $V(k)$ must satisfy the ODE:
$$\rho V(k) = \frac{[V'(k)]^{(\gamma-1)/\gamma}}{1-\gamma} + V'(k) \left[Ak^\alpha - (V'(k))^{-1/\gamma} - \delta k\right]$$

This is a nonlinear ODE that is difficult to solve analytically. Traditional approaches use finite difference methods on a discrete grid, but those methods suffer from the curse of dimensionality in higher-dimensional problems.

### Steady State

In steady state, capital is constant ($\dot{k} = 0$), so the law of motion $\dot{k} = Ak^\alpha - c - \delta k = 0$ gives us $c^* = Ak^{*\alpha} - \delta k^*$. To pin down $k^*$, note that the Euler equation in continuous time requires the marginal product of capital to equal the discount rate plus depreciation:

$$\alpha A k^{*\alpha - 1} = \rho + \delta$$

Solving for $k^*$:

$$k^* = \left(\frac{\alpha A}{\rho + \delta}\right)^{1/(1-\alpha)}$$

Steady-state consumption then follows from the resource constraint:

$$c^* = A k^{*\alpha} - \delta k^*$$

Since $\gamma > 1$, flow utility $U(c) = c^{1-\gamma}/(1-\gamma)$ is always negative, and the value function at steady state is $V(k^*) = U(c^*)/\rho$. For our calibration this works out to about $-42$, which motivates the initial guess of $-60$ for the neural network (a rough average across the state space, where $V$ is lower for $k < k^*$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from jax import grad, jit
import equinox as eqx
import optax

jax.config.update('jax_platform_name', 'cpu')

# Model parameters
gamma = 2.0      # Risk aversion (CRRA)
rho   = 0.04     # Discount rate
A     = 0.5      # TFP
alpha = 0.36     # Capital share
delta = 0.05     # Depreciation rate

# Steady state
k_ss = (alpha * A / (rho + delta))**(1 / (1 - alpha))
c_ss = A * k_ss**alpha - delta * k_ss
U_ss = c_ss**(1 - gamma) / (1 - gamma)
V_ss = U_ss / rho

print(f"Steady state capital:     k* = {k_ss:.3f}")
print(f"Steady state consumption: c* = {c_ss:.3f}")
print(f"Steady state utility:   U(c*) = {U_ss:.3f}")
print(f"Steady state value:    V(k*) = {V_ss:.1f}")

### Calibration

| Parameter | Symbol | Value | Description |
|---|---|---|---|
| Risk aversion | $\gamma$ | 2.0 | CRRA coefficient |
| Discount rate | $\rho$ | 0.04 | Rate of time preference (4% annually) |
| TFP | $A$ | 0.5 | Total factor productivity |
| Capital share | $\alpha$ | 0.36 | Returns to scale (common value for U.S. data) |
| Depreciation | $\delta$ | 0.05 | Capital depreciation rate (5% annually) |

## 2. The Neural Network Approach

Instead of solving for $V(k)$ on a discrete grid, we approximate it with a neural network $V_\theta(k)$ parameterized by weights $\theta$. The key insight is to use **automatic differentiation** to compute $V'_\theta(k)$ exactly, then use this derivative to:

1. Derive the optimal consumption policy: $c^*(k) = [V'_\theta(k)]^{-1/\gamma}$
2. Compute the HJB residual at any point $k$

### The HJB Residual

The **HJB residual** at a point $k$ is defined as:
$$\mathcal{R}(k; \theta) = U(c^*(k)) + V'_\theta(k) \cdot \mu_k(c^*(k)) - \rho V_\theta(k)$$

where:
- $c^*(k) = [V'_\theta(k)]^{-1/\gamma}$ is the optimal consumption
- $\mu_k(c) = Ak^\alpha - c - \delta k$ is the capital drift (law of motion)
- $U(c) = c^{1-\gamma}/(1-\gamma)$ is utility

### The Loss Function

The loss function is the mean squared residual over randomly sampled points:
$$\mathcal{L}(\theta) = \frac{1}{M} \sum_{j=1}^M [\mathcal{R}(k_j; \theta)]^2$$

We minimize this loss using stochastic gradient descent. At convergence, the neural network satisfies the HJB equation to high precision across the entire state space.

## 3. Network Architecture

We use a simple feedforward architecture with 3 hidden layers of 12 neurons each, tanh activation, and a linear output layer. This architecture has approximately 349 trainable parameters.

| Layer | Input Dim | Output Dim | Activation |
|---|---|---|---|
| Hidden 1 | 1 | 12 | Tanh |
| Hidden 2 | 12 | 12 | Tanh |
| Hidden 3 | 12 | 12 | Tanh |
| Output | 12 | 1 | Linear |

### Key Implementation Choices

1. **Tanh activation**: The tanh function is smooth and infinitely differentiable — crucial since we need to differentiate through the network to compute $V'_\theta(k)$.

2. **Initial guess**: The output layer bias is initialized to $-60$. Why? With $\gamma = 2$, flow utility is $U(c) = c^{1-\gamma}/(1-\gamma) = -1/c$, which is always negative. The value function — a discounted integral of these negative utilities — is therefore a large negative number across the state space. If the network starts with the default bias of zero, gradient descent has to push the output from $V \approx 0$ down to $V \approx -60$, which is slow. Setting the bias to $-60$ starts the network in the right neighborhood.

3. **Derivative clamping**: We clamp $V'_\theta(k)$ above a small threshold $\epsilon = 10^{-7}$ to avoid numerical issues when computing $c = (V')^{-1/\gamma}$ for large $\gamma$.

### From plain JAX to Equinox

In the previous notebook, we built a neural network from scratch using a dictionary of parameter arrays and a `forward` function. That works well for a single hidden layer, but here we have four layers — meaning eight parameter arrays (a weight matrix and bias vector per layer). Managing these manually becomes cumbersome, especially when we need to differentiate the loss with respect to all network weights while keeping model hyperparameters fixed.

[Equinox](https://docs.kidger.site/equinox/) is a lightweight library built on JAX that lets us define a network as a Python class. The class stores parameters as attributes and defines a `__call__` method for the forward pass — similar to PyTorch's `nn.Module`, but using JAX under the hood. Equinox also provides utilities like `eqx.filter_value_and_grad` (differentiate only the trainable parameters) and `eqx.filter_jit` (JIT-compile while handling non-array class attributes), which would require manual bookkeeping in plain JAX.

In [ ]:
class NeuralNet(eqx.Module):
    """Neural network for approximating the value function."""
    layers: list

    def __init__(self, key, nNeurons=12, initGuess=0):
        """Initialize the network.
        
        Args:
            key: JAX random key
            nNeurons: Number of neurons in hidden layers
            initGuess: Initial bias for output layer
        """
        key1, key2, key3, key4 = jax.random.split(key, 4)
        self.layers = [
            eqx.nn.Linear(1, nNeurons, key=key1),
            eqx.nn.Linear(nNeurons, nNeurons, key=key2),
            eqx.nn.Linear(nNeurons, nNeurons, key=key3),
            eqx.nn.Linear(nNeurons, 1, key=key4),
        ]
        # Set initial guess in the last layer bias
        self.layers[-1] = eqx.tree_at(
            lambda l: l.bias, self.layers[-1], jnp.array([initGuess], dtype=jnp.float32)
        )

    def __call__(self, x):
        """Forward pass: input -> hidden layers (tanh) -> output (linear)."""
        for layer in self.layers[:-1]:
            x = jnp.tanh(layer(x))        # hidden layers with Tanh activation
        x = self.layers[-1](x)             # output layer (linear)
        return x

## 4. Loss Function and Training

### Computing the HJB Error

To train the network, we need to:

1. Sample a batch of capital values $\{k_1, k_2, \ldots, k_M\}$ uniformly from the state space $[k_{\min}, k_{\max}]$
2. For each $k_j$, compute:
   - The value $V_\theta(k_j)$ (forward pass)
   - The derivative $V'_\theta(k_j)$ (automatic differentiation)
   - The HJB residual $\mathcal{R}(k_j; \theta)$
3. Compute the mean squared error across the batch
4. Use backpropagation to update the network weights

### Training Configuration

- **Batch size**: $M = 100$ random points per epoch
- **State space**: $k \in [0.1, 10]$ (covers steady state and surrounding region)
- **Optimizer**: Adam with learning rate $10^{-3}$
- **Epochs**: 100,000
- **Number of parameters**: ~349

In [ ]:
def V_and_Vprime(network, k):
    """Compute V(k) and V'(k) for a single capital value."""
    k_arr = jnp.array([k])
    V = network(k_arr)[0]
    V_prime = grad(lambda k_scalar: network(jnp.array([k_scalar]))[0])(k)
    return V, V_prime

def HJB_error(network, inputs):
    """Compute mean squared HJB error over a batch.
    
    The HJB equation is:
    rho * V(k) = max_c { U(c) + V'(k) * [A*k^alpha - c - delta*k] }
    
    At optimum, c = (V'(k))^(-1/gamma), so the error is:
    HJB_error = U(c*) + V'(k) * mu_K - rho * V(k)
    """

    def single_error(k):
        V, V_prime = V_and_Vprime(network, k)
        V_prime = jnp.maximum(V_prime, 1e-7)       # clamp derivative to avoid numerical issues

        Y = A * (k**alpha)                          # Output: Y = A * k^alpha
        C = V_prime**(-1/gamma)                      # Consumption: c = (V')^(-1/gamma)
        U = C**(1-gamma)/(1-gamma)                   # Utility: U(c) = c^(1-gamma)/(1-gamma)
        mu_K = Y - C - delta*k                       # Capital drift: dk/dt = Y - C - delta*K

        HJB = U + V_prime*mu_K - rho*V              # HJB residual
        return HJB**2

    errors = jax.vmap(single_error)(inputs)
    return jnp.mean(errors)

@eqx.filter_jit
def train_step(network, opt_state, inputs, optimizer):
    """Single training step: compute loss, gradients, and update weights."""
    loss, grads = eqx.filter_value_and_grad(HJB_error)(network, inputs)
    updates, opt_state = optimizer.update(grads, opt_state, network)
    network = eqx.apply_updates(network, updates)
    return network, opt_state, loss

In [ ]:
# Training configuration
batchSize = 100
number_epochs = 100_000
kMin, kMax = 0.1, 10.0
VFInitGuess = -60

# JAX requires explicit random keys (no global state like NumPy).
# jax.random.split(key) produces two fresh, independent keys:
# one for the current operation, one to carry forward.
key = jax.random.PRNGKey(1234)

# Initialize network
key, subkey = jax.random.split(key)
Value_net = NeuralNet(subkey, initGuess=VFInitGuess)

# Initialize optimizer and training state
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(eqx.filter(Value_net, eqx.is_array))
losses = np.zeros(number_epochs)

# Training loop
print("Training the neural network to satisfy the HJB equation...")
for epoch in range(number_epochs):
    # Split key for this epoch: subkey for sampling, key carries forward
    key, subkey = jax.random.split(key)
    inputs = jax.random.uniform(subkey, shape=(batchSize,), minval=kMin, maxval=kMax)
    
    # Training step
    Value_net, opt_state, loss = train_step(Value_net, opt_state, inputs, optimizer)
    losses[epoch] = loss.item()
    
    # Print progress
    if (epoch + 1) % 20_000 == 0:
        print(f"  Epoch {epoch+1:6d}: loss = {losses[epoch]:.2e}")

print(f"Training complete. Final loss: {losses[-1]:.2e}")

## 5. Results

We compare the neural network solution against a finite difference benchmark computed using standard numerical methods.

In [ ]:
# Load finite difference benchmark
reference_V = np.genfromtxt('BEN_V.csv')
reference_C = np.genfromtxt('BEN_C.csv')

print("Reference data loaded.")
print(f"Reference value function shape: {reference_V.shape}")
print(f"Reference policy function shape: {reference_C.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
# Smooth the loss with a running average (window size 10)
smoothed_loss = np.convolve(losses, np.ones(10)/10, mode='valid')
ax.plot(smoothed_loss, linewidth=2)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss (HJB residual²)')
ax.set_title('Training Loss Evolution')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpreting the Loss Curve

Three phases are visible in the training loss:

1. **Initial plateau** (iterations 1–100): Thanks to the warm start at $V \approx -60$, the network begins with a reasonable loss. The optimizer spends this phase finding a productive descent direction.

2. **Rapid descent** (iterations 100–10,000): The network learns the shape of the value function quickly, with the loss dropping by several orders of magnitude.

3. **Fine-tuning** (iterations 10,000–100,000): The loss approaches its floor with higher variance from batch sampling. The final loss of ~10⁻⁶ to 10⁻⁷ indicates the HJB equation is satisfied to high precision.

In [ ]:
# Evaluate neural network on a fine grid for plotting
gridSize = 10_000
K = np.linspace(kMin, kMax, gridSize)
V_nn = np.array([Value_net(jnp.array([k], dtype=jnp.float32))[0] for k in K])

# Plot comparison
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(K, V_nn, 'b:', linewidth=3, label='Neural Network')
ax.plot(K, reference_V, 'C1-', linewidth=2.5, label='Finite Difference', alpha=0.8)
ax.set_xlabel('Capital $k$', fontsize=12)
ax.set_ylabel('$V(k)$', fontsize=12)
ax.set_title('Value Function Comparison', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim(0, 6)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Properties of the Value Function

The neural network solution exhibits the expected economic properties:

1. **Negative values**: $V(k) < 0$ throughout the state space. This follows from $\gamma > 1$: flow utility $U(c) = c^{1-\gamma}/(1-\gamma)$ is always negative, so the discounted integral is too.

2. **Monotonicity**: $V'(k) > 0$, so the value function is strictly increasing in capital. More capital is always better, which makes economic sense.

3. **Concavity**: $V''(k) < 0$, consistent with diminishing returns to additional capital.

4. **Smoothness**: The neural network provides a smooth approximation, which is natural since we use infinitely differentiable tanh activations.

The close agreement between the neural network (blue dotted line) and the finite difference benchmark (orange solid line) validates the approach across the entire state space.

In [ ]:
def optimal_C(network, k):
    """Compute optimal consumption for a single capital value.
    
    From the first-order condition: U'(c) = V'(k)
    Since U'(c) = c^(-gamma), we have c = (V'(k))^(-1/gamma)
    """
    V_prime = grad(lambda k_scalar: network(jnp.array([k_scalar]))[0])(k)
    V_prime = jnp.maximum(V_prime, 1E-7)  # Clamp to avoid numerical issues
    return V_prime**(-1/gamma)

# Evaluate policy function on the grid
C_nn = np.array([optimal_C(Value_net, jnp.float32(k)) for k in K])

# Plot comparison with reference
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(K, C_nn, 'b:', linewidth=3, label='Neural Network')
ax.plot(K, reference_C, 'C1-', linewidth=2.5, label='Finite Difference', alpha=0.8)
ax.axvline(k_ss, color='grey', linestyle='--', alpha=0.6, linewidth=1.5, label=f'Steady state: $k^* = {k_ss:.2f}$')
ax.set_xlabel('Capital $k$', fontsize=12)
ax.set_ylabel('$c^*(k)$', fontsize=12)
ax.set_title('Consumption Policy Function', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim(0, 6)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Properties of the Policy Function

The optimal consumption policy exhibits the expected economic behavior:

1. **Increasing in capital**: $c^*(k)$ is strictly increasing. Households with more capital consume more in absolute terms.

2. **Concave**: The policy function is concave, meaning the marginal propensity to consume $dc/dk$ decreases with $k$. Richer households consume a smaller fraction of additional capital.

3. **Steady state**: At $k = k^*$, consumption equals the sustainable level where capital remains constant. For our parameterization, $c^*(k^*) \approx 0.66$.

4. **Bounded away from zero**: Consumption is always positive, never approaching zero, which is consistent with the CRRA utility specification.

The agreement between the neural network and finite difference solutions demonstrates that automatic differentiation successfully captures the optimal consumption policy derived from the value function gradient.

## 6. Discussion

### Key Implementation Insights

1. **Automatic differentiation**: JAX's `grad` function gives us exact $V'_\theta(k)$, which is used both to:
   - Derive the optimal policy: $c^*(k) = [V'_\theta(k)]^{-1/\gamma}$
   - Compute the HJB residual
   
   Without automatic differentiation, computing derivatives numerically would introduce discretization errors.

2. **Batch sampling**: Sampling random points uniformly from $[k_{\min}, k_{\max}]$ provides implicit regularization. This avoids overfitting to particular grid points.

3. **Initial guess**: Setting the output bias to $-60$ provides a warm start that dramatically accelerates convergence. This represents prior knowledge about the approximate scale of the value function.

4. **No grid required**: The neural network defines a smooth function everywhere on the state space, not just at discrete grid points. This enables efficient evaluation anywhere.

### Scalability and Higher Dimensions

The real power of this approach emerges in higher-dimensional problems where finite difference methods become computationally infeasible.

**Curse of dimensionality**: With $d$ state variables and $N$ grid points per dimension, finite difference methods require $O(N^d)$ evaluations. For example:
- 1D problem with $N=100$ points: $100$ evaluations
- 2D problem with $N=100$ points: $10,000$ evaluations
- 5D problem with $N=100$ points: $10$ billion evaluations

The neural network approach avoids this curse by learning a smooth function directly, with computational cost proportional to the network architecture (typically a few hundred parameters regardless of dimension) rather than the grid size.

## Summary

We have solved the continuous-time neoclassical growth model by parameterizing the value function as a neural network and minimizing the HJB residual via stochastic gradient descent. The key steps were:

1. **Parameterized** the value function as a neural network $V_\theta(k)$
2. **Computed** the derivative $V'_\theta(k)$ via automatic differentiation
3. **Computed** the HJB residual at sampled points
4. **Minimized** the mean squared residual via stochastic gradient descent

The resulting policy function closely matches the finite difference benchmark and exhibits all expected economic properties. This approach scales favorably to higher-dimensional problems and represents a promising direction for solving dynamic economic models.